# FunnyBirds CBM — Counterfactual Concept Swap

**Motivation:** recall gaps in `funnybirds_cbm_recall.ipynb` show that a concept probe
fires differently across species even when both truly have the concept. But a recall gap
is a behavioural proxy — it could reflect visual confounding in the backbone rather than
species identity leaking *through the concept bottleneck itself*.

**This notebook tests the mechanism directly** using the CBM's own bottleneck and label head:

1. For an all-positive pair (species A, species B, concept C):
   - Take species-A test images → compute concept activations z_A ∈ ℝ^26
   - Replace z_A[C] with z_B[C] (same concept, different species donor)
   - Run the label head on the modified activations
   - **If P(species B) rises**: the bottleneck value z_C was carrying species-B identity

2. **Species-identity probe on z**: train a linear classifier z → species_id.
   If concept activations alone predict species, the bottleneck is leaking.

3. **Per-concept species discriminability**: for each concept C, how accurately does
   z_C (a single scalar) distinguish species A from B?

FunnyBirds is ideal for this: the class-concept matrix is exact, so we know exactly
which species should have which concept, and the label head maps 26 → 50 with no noise.

In [ ]:
import json
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

## 0. Configuration

In [ ]:
ROOT = Path('/scratch/network/cr7998/cv_emergence_project')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FB          = ROOT / 'data' / 'FunnyBirds'
CBM_FEATS   = ROOT / 'features' / 'resnet50_cbm_funnybirds'
CBM_CKPT    = ROOT / 'checkpoints_funnybirds' / 'cbm_funnybirds.pth'

# Recall gap results from funnybirds_cbm_recall.ipynb (for correlation analysis)
RECALL_GAP_CSV = Path('fb_baseline_vs_cbm_table.csv')  # optional
CBM_SPECIES_CSV = Path('fb_cbm_species.csv')            # optional

N_SPECIES   = 50
N_CONCEPTS  = 26
SWAP_SEED   = 0
PROBE_EPOCHS = 30   # for species-identity and per-concept probes

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

assert FB.exists(),        f'Missing FunnyBirds root: {FB}'
assert CBM_FEATS.exists(), f'Missing CBM features: {CBM_FEATS}'
assert CBM_CKPT.exists(),  f'Missing CBM checkpoint: {CBM_CKPT}'
print(f'[config] device: {device}')
print(f'[config] CBM checkpoint: {CBM_CKPT}')

## 1. Load metadata, concept names, and class-concept matrix

In [ ]:
from datasets.funnybirds_dataset import (
    FunnyBirdsDataset, concept_names as _cnames, NUM_CONCEPTS
)

CONCEPT_NAMES = _cnames()  # list of 26 strings, e.g. ['beak_0', 'beak_1', ...]
CONCEPT_TO_IDX = {c: i for i, c in enumerate(CONCEPT_NAMES)}

# Part grouping
PART_GROUPS = {
    part: [i for i, c in enumerate(CONCEPT_NAMES) if c.startswith(f'{part}_')]
    for part in ['beak', 'wing', 'tail', 'foot', 'eye']
}
PART_COLORS = {'beak': 'steelblue', 'wing': 'seagreen', 'tail': 'crimson',
               'foot': 'darkorange', 'eye': 'purple'}
CONCEPT_TO_PART = {c: part for part, idxs in PART_GROUPS.items()
                   for c in [CONCEPT_NAMES[i] for i in idxs]}

# Exact class-concept matrix: cc[s, c] = 1 iff species s truly has concept c
_fb_ds = FunnyBirdsDataset(FB, split='train')
cc_matrix, _ = _fb_ds.get_class_concept_matrix()  # [50, 26] int tensor
cc_matrix = cc_matrix.numpy()                      # numpy for indexing

cc_df = pd.DataFrame(
    cc_matrix,
    columns=CONCEPT_NAMES,
    index=[f'funnybird_{i:02d}' for i in range(N_SPECIES)],
)
print(f'Concepts ({len(CONCEPT_NAMES)}): {CONCEPT_NAMES}')
print(f'CC matrix: {cc_matrix.shape}  — each row sums to {cc_matrix.sum(1).mean():.0f}')
cc_df.head()

In [ ]:
# Load per-image metadata (species_id, is_train)
meta = pd.read_csv(FB / 'metadata' / 'images.csv')
id2name = dict(pd.read_csv(FB / 'metadata' / 'classes.csv')
               .pipe(lambda d: zip(d['class_id'], d['class_name'])))
meta['species_name'] = meta['class_id'].map(id2name)
meta_test = meta[meta['is_train'] == 0].copy()

# Map image_id → species_id for fast lookup
imgid_to_sid = dict(zip(meta['image_id'], meta['class_id']))

print(f'Test images: {len(meta_test)}  ({meta_test["class_id"].nunique()} species)')

## 2. Load CBM weights and compute concept activations

Architecture: `backbone → avgpool [2048] → concept_head [26] → sigmoid → label_head [50]`

We load:
- **avgpool features** — already extracted at `features/resnet50_cbm_funnybirds/avgpool_{split}.pt`
- **concept_head weights** — from checkpoint, to reconstruct z = sigmoid(avgpool @ W_c.T + b_c)
- **label_head weights** — from checkpoint, to run swapped activations through the classifier

In [ ]:
def _load(p):
    try:
        return torch.load(p, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(p, map_location='cpu')

ckpt = _load(CBM_CKPT)
sd   = ckpt['model_state_dict']

W_c = sd['concept_head.weight'].float()  # [26, 2048]
b_c = sd['concept_head.bias'].float()   # [26]
W_y = sd['label_head.weight'].float()   # [50, 26]
b_y = sd['label_head.bias'].float()     # [50]

print(f'concept_head: {W_c.shape}  bias: {b_c.shape}')
print(f'label_head:   {W_y.shape}  bias: {b_y.shape}')
cfg = ckpt.get('config', {})
print(f'config: {cfg}')

In [ ]:
def load_avgpool(split):
    p = CBM_FEATS / f'avgpool_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    X = _load(p)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    return X.float()


def load_image_ids(split):
    p = CBM_FEATS / f'labels_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    t = _load(p)
    ids = t['image_ids'] if isinstance(t, dict) else t
    if isinstance(ids, torch.Tensor):
        ids = ids.cpu().numpy()
    return np.asarray(ids).reshape(-1).astype(int)


@torch.no_grad()
def compute_concept_activations(avgpool_feats, W_c, b_c):
    """z = sigmoid(avgpool @ W_c.T + b_c)  →  [N, 26]"""
    logits = avgpool_feats @ W_c.T + b_c
    return torch.sigmoid(logits)


@torch.no_grad()
def compute_species_logits(z, W_y, b_y):
    """logits = z @ W_y.T + b_y  →  [N, 50]"""
    return z @ W_y.T + b_y


# ── Load test split ───────────────────────────────────────────────────────────
avg_te  = load_avgpool('test')
ids_te  = load_image_ids('test')
z_te    = compute_concept_activations(avg_te, W_c, b_c)      # [N_te, 26]
logits_te = compute_species_logits(z_te, W_y, b_y)            # [N_te, 50]
sids_te = np.array([imgid_to_sid[int(i)] for i in ids_te])    # [N_te] species IDs

# ── Load train split (for species probe training) ─────────────────────────────
avg_tr  = load_avgpool('train')
ids_tr  = load_image_ids('train')
z_tr    = compute_concept_activations(avg_tr, W_c, b_c)       # [N_tr, 26]
sids_tr = np.array([imgid_to_sid[int(i)] for i in ids_tr])    # [N_tr] species IDs

print(f'Test:  avgpool {avg_te.shape}  →  z {z_te.shape}')
print(f'Train: avgpool {avg_tr.shape}  →  z {z_tr.shape}')
print(f'Species ID range: {sids_te.min()}..{sids_te.max()}')

In [ ]:
# Sanity check: CBM species accuracy on test set
pred_species = logits_te.argmax(dim=1).numpy()
acc = (pred_species == sids_te).mean()
print(f'CBM test accuracy (species): {acc:.4f}')

# Sanity check: concept accuracy vs ground truth
# cc_matrix[sid, concept] = ground truth for species sid
gt_concepts_te = cc_matrix[sids_te]                    # [N_te, 26]
pred_concepts  = (z_te.numpy() > 0.5).astype(int)      # [N_te, 26]
concept_acc    = (pred_concepts == gt_concepts_te).mean()
print(f'CBM test accuracy (concept, threshold=0.5): {concept_acc:.4f}')

# Per-concept accuracy
per_concept_acc = (pred_concepts == gt_concepts_te).mean(axis=0)
for name, acc_c in zip(CONCEPT_NAMES, per_concept_acc):
    print(f'  {name:12s}: {acc_c:.3f}')

## 3. Counterfactual concept swap

For an all-positive pair (species A, species B, concept C):

```
z_swapped = z_A  but  z_swapped[:, C] ← z_B[:, C]
logits_swapped = z_swapped @ W_y.T + b_y
shift_B = mean(softmax(logits_swapped)[:, B]) − mean(softmax(logits_A)[:, B])
```

**Interpretation:**
- `shift_B > 0` → swapping z_C from B into A *raised* P(species B). z_C was carrying species-B signal.
- `shift_A < 0` → P(species A) fell. Both effects together confirm leakage.
- `leakage_score = shift_B − shift_A` — signed, larger = more species identity in z_C.

We do both directions (A→B swap and B→A swap) and average for symmetry.

In [ ]:
@torch.no_grad()
def concept_swap(
    z_A: torch.Tensor,    # [N_A, K]  recipient activations
    z_B: torch.Tensor,    # [N_B, K]  donor activations
    concept_idx: int,
    W_y: torch.Tensor,    # [n_species, K]
    b_y: torch.Tensor,    # [n_species]
):
    """
    Replace z_A[:, concept_idx] with z_B[:, concept_idx] for all donor-recipient combos.
    For each recipient A-image, average the swap over all N_B donor B-images.

    Returns:
        p_orig  [N_A, n_species]  softmax probabilities before swap
        p_swap  [N_A, n_species]  softmax probabilities after swap (averaged over donors)
    """
    N_A, K = z_A.shape
    N_B    = z_B.shape[0]

    # Original predictions for A
    p_orig = torch.softmax(z_A @ W_y.T + b_y, dim=-1)   # [N_A, n_species]

    # Build all (A, B) combos: [N_A, N_B, K], swap concept slot
    z_swap = z_A.unsqueeze(1).expand(N_A, N_B, K).clone()   # [N_A, N_B, K]
    z_swap[:, :, concept_idx] = z_B[:, concept_idx].unsqueeze(0)  # broadcast donors

    logits_swap = z_swap.view(N_A * N_B, K) @ W_y.T + b_y  # [N_A*N_B, n_species]
    p_swap_all  = torch.softmax(logits_swap, dim=-1).view(N_A, N_B, -1)
    p_swap      = p_swap_all.mean(dim=1)                    # [N_A, n_species]

    return p_orig, p_swap


def pair_swap_metrics(sid_A, sid_B, concept_idx, z_te, sids_te, W_y, b_y):
    """
    Compute counterfactual swap metrics for one (species-A, species-B, concept-C) triple.
    Does both A→B and B→A swaps and returns the average (symmetric).
    """
    mask_A = (sids_te == sid_A)
    mask_B = (sids_te == sid_B)
    z_A = z_te[mask_A]   # [N_A, K]
    z_B = z_te[mask_B]   # [N_B, K]

    if len(z_A) == 0 or len(z_B) == 0:
        return None

    # A→B direction: put B's z_C into A
    p_orig_A, p_swap_A = concept_swap(z_A, z_B, concept_idx, W_y, b_y)
    shift_B_fwd = (p_swap_A[:, sid_B] - p_orig_A[:, sid_B]).mean().item()  # should be +
    shift_A_fwd = (p_swap_A[:, sid_A] - p_orig_A[:, sid_A]).mean().item()  # should be -

    # B→A direction: put A's z_C into B
    p_orig_B, p_swap_B = concept_swap(z_B, z_A, concept_idx, W_y, b_y)
    shift_A_bwd = (p_swap_B[:, sid_A] - p_orig_B[:, sid_A]).mean().item()  # should be +
    shift_B_bwd = (p_swap_B[:, sid_B] - p_orig_B[:, sid_B]).mean().item()  # should be -

    # Leakage score per direction, then average
    leakage_fwd = shift_B_fwd - shift_A_fwd   # >0 = B signal in A's z_C
    leakage_bwd = shift_A_bwd - shift_B_bwd   # >0 = A signal in B's z_C
    leakage_sym = 0.5 * (leakage_fwd + leakage_bwd)

    return {
        'sid_A': int(sid_A), 'sid_B': int(sid_B),
        'concept_idx': int(concept_idx), 'concept': CONCEPT_NAMES[concept_idx],
        'shift_B_fwd': float(shift_B_fwd),  # P(B) change when z_C swapped A→B
        'shift_A_fwd': float(shift_A_fwd),  # P(A) change when z_C swapped A→B
        'shift_A_bwd': float(shift_A_bwd),  # P(A) change when z_C swapped B→A
        'shift_B_bwd': float(shift_B_bwd),  # P(B) change when z_C swapped B→A
        'leakage_fwd': float(leakage_fwd),
        'leakage_bwd': float(leakage_bwd),
        'leakage_sym': float(leakage_sym),  # main metric
        'n_A': int(mask_A.sum()),
        'n_B': int(mask_B.sum()),
    }


print('Defined: concept_swap  pair_swap_metrics')

## 4. Run swaps for all GT-positive pairs

For each concept C: find all species that truly have C=1 (from class-concept matrix),
enumerate all pairs among them, run the swap.

In [ ]:
swap_rows = []

for c_idx, c_name in enumerate(CONCEPT_NAMES):
    # Species that GT-have this concept
    positive_sids = np.where(cc_matrix[:, c_idx] == 1)[0]
    n_pos = len(positive_sids)
    if n_pos < 2:
        continue

    for sid_A, sid_B in combinations(positive_sids, 2):
        result = pair_swap_metrics(
            sid_A, sid_B, c_idx, z_te, sids_te, W_y, b_y
        )
        if result is not None:
            swap_rows.append(result)

swap_df = pd.DataFrame(swap_rows)
swap_df['part'] = swap_df['concept'].map(CONCEPT_TO_PART)

print(f'Swap results: {len(swap_df)} rows  ({swap_df["concept"].nunique()} concepts)')
print(f'leakage_sym  mean={swap_df["leakage_sym"].mean():.4f}  '
      f'median={swap_df["leakage_sym"].median():.4f}  '
      f'max={swap_df["leakage_sym"].max():.4f}')
swap_df.head(10)

In [ ]:
swap_df.to_csv('fb_cbm_counterfactual_swap.csv', index=False)
print('Saved fb_cbm_counterfactual_swap.csv')

## 5. Per-concept and per-part aggregation

In [ ]:
concept_agg = (
    swap_df.groupby(['concept', 'part'], as_index=False)
    .agg(
        n_pairs      = ('leakage_sym', 'size'),
        leakage_mean = ('leakage_sym', 'mean'),
        leakage_std  = ('leakage_sym', 'std'),
        leakage_max  = ('leakage_sym', 'max'),
        shift_B_mean = ('shift_B_fwd', 'mean'),   # mean P(B) increase
        shift_A_mean = ('shift_A_fwd', 'mean'),   # mean P(A) change (should be neg)
        frac_positive = ('leakage_sym', lambda s: (s > 0).mean()),
    )
    .sort_values('leakage_mean', ascending=False)
    .reset_index(drop=True)
)
print('Per-concept leakage summary:')
display(concept_agg)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Panel A: leakage_mean per concept, coloured by part
ax = axes[0]
colors = [PART_COLORS[p] for p in concept_agg['part']]
bars = ax.barh(concept_agg['concept'], concept_agg['leakage_mean'], color=colors, alpha=0.8)
ax.errorbar(
    concept_agg['leakage_mean'],
    range(len(concept_agg)),
    xerr=concept_agg['leakage_std'].fillna(0),
    fmt='none', color='black', alpha=0.5, capsize=3,
)
ax.axvline(0, color='gray', ls='--', alpha=0.7)
ax.set_xlabel('Mean leakage score (shift_B − shift_A, symmetric)')
ax.set_title('Counterfactual swap leakage by concept\n(>0 = concept activation carries species identity)')
# Part legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=p) for p, c in PART_COLORS.items()],
          fontsize=8, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)

# Panel B: per-part aggregation
ax = axes[1]
part_agg = swap_df.groupby('part')['leakage_sym'].agg(['mean','sem','size']).reset_index()
part_agg = part_agg.sort_values('mean', ascending=False)
ax.barh(part_agg['part'],
        part_agg['mean'],
        xerr=part_agg['sem'],
        color=[PART_COLORS[p] for p in part_agg['part']],
        alpha=0.8, capsize=4)
for _, r in part_agg.iterrows():
    ax.text(max(r['mean'], 0) + 0.001, list(part_agg['part']).index(r['part']),
            f"n={int(r['size'])}", va='center', fontsize=8)
ax.axvline(0, color='gray', ls='--', alpha=0.7)
ax.set_xlabel('Mean leakage score (± SEM)')
ax.set_title('Leakage by body part')
ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('FunnyBirds CBM: Counterfactual concept swap — leakage scores', y=1.02)
plt.tight_layout()
plt.savefig('fb_cbm_swap_leakage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_swap_leakage.png')

In [ ]:
# Decompose leakage: P(B) increase vs P(A) decrease after A→B swap
fig, ax = plt.subplots(figsize=(7, 5))

for _, row in concept_agg.iterrows():
    color = PART_COLORS[row['part']]
    ax.scatter(row['shift_B_mean'], -row['shift_A_mean'],
               color=color, s=60, alpha=0.8, zorder=3)
    ax.annotate(row['concept'],
                (row['shift_B_mean'], -row['shift_A_mean']),
                fontsize=6, alpha=0.7)

ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.axvline(0, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('ΔP(species B)  after swapping z_C from B into A  (should be >0 if leakage)')
ax.set_ylabel('−ΔP(species A)  (should be >0 if leakage)')
ax.set_title('Leakage decomposition per concept\n'
             '(top-right quadrant = both effects consistent with leakage)')
ax.legend(handles=[Patch(color=c, label=p) for p, c in PART_COLORS.items()], fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fb_cbm_swap_decomp.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_swap_decomp.png')

## 6. Species-identity probe on z

Train a linear classifier `z → species_id` on the CBM bottleneck activations.
High accuracy means the 26-dimensional concept bottleneck encodes species identity —
leakage is in the representation itself, not just the downstream probe.

Compare against a random-baseline (26-dim Gaussian noise → species_id) to
calibrate what 'chance' looks like.

In [ ]:
import random

class LinearProbe(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_classes)

    def forward(self, x):
        return self.fc(x)


def train_classifier(
    X_tr, y_tr, X_te, y_te,
    n_classes, seed=0, lr=1e-2, wd=1e-4, epochs=30, batch=512
):
    """Train linear multiclass probe, return test accuracy."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    X_tr = X_tr.to(device); X_te = X_te.to(device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long, device=device)
    y_te_t = torch.tensor(y_te, dtype=torch.long, device=device)

    probe = LinearProbe(X_tr.shape[1], n_classes).to(device)
    opt   = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.CrossEntropyLoss()

    n = X_tr.shape[0]
    for _ in range(epochs):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            loss = loss_fn(probe(X_tr[idx]), y_tr_t[idx])
            opt.zero_grad(); loss.backward(); opt.step()

    with torch.no_grad():
        probe.eval()
        preds = probe(X_te).argmax(dim=1)
        acc   = float((preds == y_te_t).float().mean().item())
    return acc


print('Defined: train_classifier')

In [ ]:
# Full z (26-dim) → species probe
acc_full = train_classifier(
    z_tr, sids_tr, z_te, sids_te, n_classes=N_SPECIES,
    epochs=PROBE_EPOCHS,
)
print(f'Species probe on z (all 26 concepts): {acc_full:.4f}')
print(f'  chance level: {1/N_SPECIES:.4f}  ({N_SPECIES} classes)')

# Random baseline: same shape, Gaussian noise
torch.manual_seed(42)
acc_rand = train_classifier(
    torch.randn_like(z_tr), sids_tr,
    torch.randn_like(z_te), sids_te,
    n_classes=N_SPECIES, epochs=PROBE_EPOCHS,
)
print(f'Species probe on random baseline:    {acc_rand:.4f}')

# avgpool features baseline (to compare: how much species info in backbone vs bottleneck)
acc_avgpool = train_classifier(
    avg_tr, sids_tr, avg_te, sids_te, n_classes=N_SPECIES,
    epochs=PROBE_EPOCHS,
)
print(f'Species probe on avgpool features:   {acc_avgpool:.4f}')

In [ ]:
# Per-concept species probe: z[:, c:c+1] → species_id
# A single concept activation is a scalar per image — how much species info does it carry?

per_concept_species_acc = {}
for c_idx, c_name in enumerate(CONCEPT_NAMES):
    acc_c = train_classifier(
        z_tr[:, c_idx:c_idx+1], sids_tr,
        z_te[:, c_idx:c_idx+1], sids_te,
        n_classes=N_SPECIES, epochs=PROBE_EPOCHS,
    )
    per_concept_species_acc[c_name] = acc_c

probe_acc_df = pd.DataFrame([
    {'concept': c, 'species_acc': a, 'part': CONCEPT_TO_PART[c]}
    for c, a in per_concept_species_acc.items()
]).sort_values('species_acc', ascending=False)

print(f'\nPer-concept species decodability (chance={1/N_SPECIES:.3f}):')
display(probe_acc_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Panel A: per-concept species probe accuracy
ax = axes[0]
colors = [PART_COLORS[p] for p in probe_acc_df['part']]
ax.barh(probe_acc_df['concept'], probe_acc_df['species_acc'], color=colors, alpha=0.8)
ax.axvline(1/N_SPECIES, color='red', ls='--', alpha=0.7, label=f'chance (1/{N_SPECIES})')
ax.axvline(acc_full,    color='black', ls='-', alpha=0.5, label=f'all-z probe ({acc_full:.3f})')
ax.set_xlabel('Linear probe accuracy: z_C → species_id')
ax.set_title('Species-identity decodability per concept activation\n'
             '(above chance = concept bottleneck leaks species identity)')
ax.legend(fontsize=8)
ax.legend(handles=[
    *[Patch(color=c, label=p) for p, c in PART_COLORS.items()],
    plt.Line2D([0],[0], color='red', ls='--', label=f'chance 1/{N_SPECIES}'),
    plt.Line2D([0],[0], color='black', label=f'all-z ({acc_full:.3f})'),
], fontsize=7)
ax.grid(True, axis='x', alpha=0.3)

# Panel B: summary bar — backbone avgpool vs z vs random
ax = axes[1]
labels = ['Random\n(baseline)', 'CBM z\n(all 26)', 'avgpool\n(backbone)']
accs   = [acc_rand, acc_full, acc_avgpool]
ax.bar(labels, accs, color=['lightgray', 'steelblue', 'darkorange'], alpha=0.85)
ax.axhline(1/N_SPECIES, color='red', ls='--', alpha=0.7, label=f'chance 1/{N_SPECIES}')
for i, a in enumerate(accs):
    ax.text(i, a + 0.005, f'{a:.3f}', ha='center', fontsize=9)
ax.set_ylabel('Species prediction accuracy')
ax.set_title('How much species identity is in each representation?')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, min(1.0, max(accs) * 1.2))

plt.suptitle('FunnyBirds CBM: Species-identity decodability from concept bottleneck', y=1.02)
plt.tight_layout()
plt.savefig('fb_cbm_species_probe.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_species_probe.png')

## 7. Per-pair species discriminability in z_C

For each all-positive pair (A, B, concept C): train a **binary** classifier
`z_C → {species A, species B}` using only images from those two species.

Accuracy >> 0.5 means z_C alone distinguishes the two species — direct evidence
that this concept activation carries per-species signal beyond just whether
the concept is present.

In [ ]:
def binary_species_disc(sid_A, sid_B, concept_idx, z_tr, sids_tr, z_te, sids_te,
                        epochs=PROBE_EPOCHS):
    """
    Train z[:, concept_idx] → {0=A, 1=B} binary classifier.
    Uses only images from species A and B.
    Returns test accuracy (0.5 = chance).
    """
    mask_tr = np.isin(sids_tr, [sid_A, sid_B])
    mask_te = np.isin(sids_te, [sid_A, sid_B])

    if mask_tr.sum() < 4 or mask_te.sum() < 2:
        return np.nan

    X_tr = z_tr[mask_tr, concept_idx:concept_idx+1]  # [N_tr_sub, 1]
    X_te = z_te[mask_te, concept_idx:concept_idx+1]  # [N_te_sub, 1]
    y_tr = (sids_tr[mask_tr] == sid_B).astype(int)    # 0=A, 1=B
    y_te = (sids_te[mask_te] == sid_B).astype(int)

    return train_classifier(X_tr, y_tr, X_te, y_te, n_classes=2, epochs=epochs)


disc_rows = []
for c_idx, c_name in enumerate(CONCEPT_NAMES):
    positive_sids = np.where(cc_matrix[:, c_idx] == 1)[0]
    if len(positive_sids) < 2:
        continue
    for sid_A, sid_B in combinations(positive_sids, 2):
        acc = binary_species_disc(
            sid_A, sid_B, c_idx, z_tr, sids_tr, z_te, sids_te
        )
        if not np.isnan(acc):
            disc_rows.append({
                'sid_A': int(sid_A), 'sid_B': int(sid_B),
                'concept': c_name, 'concept_idx': c_idx,
                'part': CONCEPT_TO_PART[c_name],
                'disc_acc': float(acc),
            })

disc_df = pd.DataFrame(disc_rows)
disc_df['disc_above_chance'] = disc_df['disc_acc'] > 0.5

print(f'Binary discriminability: {len(disc_df)} pair-concept combos')
print(f'  mean disc_acc={disc_df["disc_acc"].mean():.4f}  '
      f'frac>0.5: {disc_df["disc_above_chance"].mean():.3f}')

disc_concept = (
    disc_df.groupby(['concept','part'])[['disc_acc','disc_above_chance']]
    .mean().reset_index()
    .sort_values('disc_acc', ascending=False)
)
display(disc_concept)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

ax = axes[0]
colors = [PART_COLORS[p] for p in disc_concept['part']]
ax.barh(disc_concept['concept'], disc_concept['disc_acc'], color=colors, alpha=0.8)
ax.axvline(0.5, color='red', ls='--', alpha=0.7, label='chance (0.5)')
ax.set_xlabel('Mean binary accuracy: z_C → {species A, species B}')
ax.set_title('Per-pair species discriminability in z_C\n'
             '(averaged over all GT-positive pairs for each concept)')
ax.legend(handles=[
    *[Patch(color=c, label=p) for p, c in PART_COLORS.items()],
    plt.Line2D([0],[0], color='red', ls='--', label='chance 0.5'),
], fontsize=7)
ax.grid(True, axis='x', alpha=0.3)

ax = axes[1]
part_disc = (
    disc_df.groupby('part')['disc_acc']
    .agg(['mean','sem'])
    .reset_index()
    .sort_values('mean', ascending=False)
)
ax.barh(part_disc['part'],
        part_disc['mean'],
        xerr=part_disc['sem'],
        color=[PART_COLORS[p] for p in part_disc['part']],
        alpha=0.8, capsize=4)
ax.axvline(0.5, color='red', ls='--', alpha=0.7, label='chance 0.5')
ax.set_xlabel('Mean discriminability (± SEM)')
ax.set_title('Discriminability by body part')
ax.legend(); ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('FunnyBirds CBM: Species discriminability in z_C (per-concept scalar)', y=1.02)
plt.tight_layout()
plt.savefig('fb_cbm_binary_disc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_binary_disc.png')

## 8. Three-level evidence: correlation across measures

The three analyses measure the same underlying phenomenon at different levels:

| Analysis | Level | Measure |
|---|---|---|
| Recall gap | Behavioural | Probe fires differently across species |
| Counterfactual swap | Causal | Replacing z_C shifts species prediction |
| Binary discriminability | Mechanistic | z_C alone separates two species |

If they agree per-concept, the evidence for species-identity leakage is robust.

In [ ]:
# Merge all three concept-level measures
merged = (
    concept_agg[['concept','part','leakage_mean']]
    .merge(disc_concept[['concept','disc_acc']], on='concept', how='inner')
    .merge(probe_acc_df[['concept','species_acc']], on='concept', how='inner')
)

# Load recall gaps if available
if CBM_SPECIES_CSV.exists():
    sp_csv = pd.read_csv(CBM_SPECIES_CSV)
    if 'attr' in sp_csv.columns and 'recall' in sp_csv.columns:
        recall_gap = (
            sp_csv.groupby('attr')['recall']
            .apply(lambda x: x.max() - x.min())
            .reset_index(name='recall_range')
            .rename(columns={'attr': 'concept'})
        )
        merged = merged.merge(recall_gap, on='concept', how='left')
        print('Merged recall_range from fb_cbm_species.csv')
    else:
        merged['recall_range'] = np.nan
        print('[warn] fb_cbm_species.csv found but lacks attr/recall columns')
else:
    merged['recall_range'] = np.nan
    print('[info] fb_cbm_species.csv not found — run funnybirds_cbm_recall.ipynb first')

display(merged)

In [ ]:
has_recall = merged['recall_range'].notna().any()
ncols = 3 if has_recall else 2
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4.5))

pairs_to_plot = [
    ('leakage_mean', 'disc_acc',    'Leakage score vs Discriminability'),
    ('leakage_mean', 'species_acc', 'Leakage score vs Species probe acc'),
]
if has_recall:
    pairs_to_plot.append(('recall_range', 'leakage_mean', 'Recall range vs Leakage score'))

for ax, (x_col, y_col, title) in zip(axes, pairs_to_plot):
    sub = merged[[x_col, y_col, 'concept', 'part']].dropna()
    colors_sub = [PART_COLORS[p] for p in sub['part']]
    ax.scatter(sub[x_col], sub[y_col], c=colors_sub, s=60, alpha=0.8, zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row['concept'], (row[x_col], row[y_col]), fontsize=6, alpha=0.6)
    if len(sub) > 2:
        r, p = stats.pearsonr(sub[x_col], sub[y_col])
        m, b = np.polyfit(sub[x_col], sub[y_col], 1)
        xs = np.array([sub[x_col].min(), sub[x_col].max()])
        ax.plot(xs, m * xs + b, 'k--', alpha=0.5, label=f'r={r:.2f}, p={p:.3f}')
        ax.legend(fontsize=8)
    ax.set_xlabel(x_col); ax.set_ylabel(y_col)
    ax.set_title(title); ax.grid(True, alpha=0.3)

plt.suptitle('FunnyBirds CBM: Cross-measure correlation (three levels of evidence)', y=1.02)
plt.tight_layout()
plt.savefig('fb_cbm_evidence_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fb_cbm_evidence_correlation.png')

## 9. Summary

### What we measured

| Measure | Value | Interpretation |
|---|---|---|
| Species probe on z (all 26) | see above | How much species info survives bottleneck |
| Species probe on avgpool (backbone) | see above | Upstream ceiling |
| Mean leakage score (swap) | see above | Causal: z_C carries species signal |
| Mean binary discriminability | see above | z_C scalar separates species |

### Interpreting leakage_score

- `leakage_sym > 0` for a concept means: swapping z_C from species B into species A
  consistently raised P(B) and lowered P(A). The bottleneck value for that concept
  is not purely concept-predictive — it also carries which species produced it.

- A concept with **high leakage + high discriminability + high species probe acc**
  is the strongest case: three independent analyses agree it leaks species identity.

### Next: compare CBM vs MCBM

Run this notebook against MCBM checkpoints at each γ value. The IB penalty should
reduce `leakage_sym` and `disc_acc` while preserving concept accuracy — that is
the direct mechanistic test of the backwash hypothesis.